In [ ]:
from perlin_numpy import generate_perlin_noise_3d, generate_perlin_noise_2d


np.random.seed(0)
noise = generate_perlin_noise_2d(
    (128, 128), (32, 32), tileable=(True, False)
)
plt.imshow(noise, cmap='gray')

import numpy as np
import matplotlib.pyplot as plt
from perlin_numpy import generate_perlin_noise_2d

def generate_ensemble_perturbation(shape=(720, 1440), seed=None):
    """
    shape: (ny, nx) 출력 격자 크기 (위도 × 경도)
    seed : 난수 시드 (재현성 확보용)
    
    각 옥타브별 조건:
        - scale: 0.2, 0.1, 0.05
        - periods: 12, 24, 48
    """
    if seed is not None:
        np.random.seed(seed)

    scales = [0.2, 0.1, 0.05]
    periods = [12, 24, 48]

    perturbation = np.zeros(shape, dtype=np.float32)

    for scale, per in zip(scales, periods):
        res = (per, per)
        noise = generate_perlin_noise_2d(shape, res)
        perturbation += scale * noise

    return perturbation


def generate_ensemble_members(n_members=10, shape=(256, 256)):
    """
    여러 ensemble 멤버를 생성
    """
    ensemble = []
    for i in range(n_members):
        member = generate_ensemble_perturbation(shape=shape, seed=i)
        ensemble.append(member)
    return np.stack(ensemble)


if __name__ == "__main__":
    # ensemble 멤버 3개 생성 예시
    ensemble = generate_ensemble_members(n_members=3, shape=(720, 1440))

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for i, ax in enumerate(axes):
        im = ax.imshow(ensemble[i], cmap="RdBu_r")
        ax.set_title(f"Member {i+1}")
        ax.axis("off")
    plt.colorbar(im, ax=axes, orientation="vertical", fraction=0.02)
    plt.suptitle("Ensemble Perturbations (3 Octaves Perlin Noise)")
    plt.show()
